# Did A really beat B? Eval variance across repeated runs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/report-evaluation-variation-across-repeated-agent-runs/eval_variance_v1.ipynb)

Companion to https://profrod.ai/articles/report-evaluation-variation-across-repeated-agent-runs. 100 AIME problems, three systems (GPT-6 Luna, the same model with a reworded prompt, GPT-5.6 Luna; reasoning effort medium), ten runs each: 3,000 recorded attempts. Part 1 recomputes the article's numbers from them. Part 2 reruns a small version on your own key.

In [ ]:
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/report-evaluation-variation-across-repeated-agent-runs/"
FILES = [
    "variance_stats_v1.py",
    "eval_variance_v1.py",
    "data/aime-tasks-v1.jsonl",
    "results/attempts-v1.jsonl",
    "results/summary-v1.json",
    "receipts/eval-variance-v1.json",
]
for name in FILES:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
print("ready")

## Part 1: recompute the article's numbers

In [ ]:
import variance_stats_v1 as V
from eval_variance_v1 import COMPARISONS, outcomes_by_system

data = outcomes_by_system()
for name, outcomes in data.items():
    rates = V.single_run_rates(outcomes)
    print(
        f"{name:20s} pass {sum(map(sum, outcomes)) / 1000:.3f}  single runs {min(rates):.2f} to {max(rates):.2f}  "
        f"naive SE {V.naive_se(outcomes):.4f}  clustered SE {V.clustered_se(outcomes):.4f}  ICC {V.icc(outcomes):.2f}"
    )

In [ ]:
for base, other in COMPARISONS:
    d = V.paired_difference(data[base], data[other])
    lo, hi = V.bootstrap_difference(data[base], data[other])
    print(f"{other} minus {base}: {d['mean']:+.3f}  (bootstrap 95% {lo:+.3f} to {hi:+.3f})")

In [ ]:
# pass@k and pass^k from per-problem success counts, against the equal-difficulty curves.
for name, outcomes in data.items():
    p = sum(map(sum, outcomes)) / 1000
    print(
        name,
        "pass^10",
        round(V.mean_curve(outcomes, 10, V.pass_hat_k), 3),
        "vs p^10",
        round(p**10, 3),
        "| pass@10",
        round(V.mean_curve(outcomes, 10, V.pass_at_k), 3),
        "vs 1-(1-p)^10",
        round(1 - (1 - p) ** 10, 3),
    )

## Part 2: rerun on your own key (optional)

Add `OPENAI_API_KEY` in Colab's Secrets panel. This reruns 20 problems, 3 runs each (180 requests, well under a dollar at the recorded prices). Your numbers will differ from the recorded ones: that is the point of the article.

In [ ]:
try:
    from google.colab import userdata
except ImportError:  # outside Colab: set OPENAI_API_KEY in the environment instead
    userdata = None
if userdata is not None:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.makedirs("mine/data", exist_ok=True)
os.makedirs("mine/results", exist_ok=True)
os.makedirs("mine/receipts", exist_ok=True)
!cp eval_variance_v1.py variance_stats_v1.py mine/
!cd mine && sed -i 's/TASKS, RUNS = 100, 10/TASKS, RUNS = 20, 3/' eval_variance_v1.py && python eval_variance_v1.py all --max-usd 1